# Panorama Image Stitching

**CSC14120 Parallel Programming · Group 16 · Final Project Report**

The report follows the [assignment structure](docs/specs/CSC14120_2026_Final%20Project.md#52-notebook-report-structure). For the submission run, open this notebook in a fresh Google Colab GPU session with a Tesla T4 and select **Runtime → Run all**. The current report includes runnable evidence from earlier verified experiments. The live final-run cells and their artifact inputs are identified at their insertion points.

### Contents

1. [Team Information](#team-information)
2. [Introduction](#introduction)
3. [Chosen Methods](#chosen-methods)
4. [V0 Sequential](#v0-sequential)
5. [V1 Basic CUDA](#v1-basic-cuda)
6. [Optimized Versions](#optimized-versions)
7. [Final Comparison](#final-comparison)
8. [Results Gallery](#results-gallery)
9. [Conclusion](#conclusion)
10. [References](#references)

<a id="team-information"></a>

## 1. Team Information

| Member | Student ID | V0 and V1 | Optimized versions and final deliverables |
| --- | --- | --- | --- |
| Nguyễn Hồ Anh Tuấn (team lead) | 23120185 | S1 preprocessing | S5 kernels in V2 and V3, V4 integration acceptance, slides and presentation material |
| Trần Hữu Kim Thành | 23120166 | S2 detection and S3 description | V2/V3/V4 pipeline and CLI integration, V3 acceptance execution, final notebook data and visualizations |
| Lê Thượng Đế | 23120232 | S4 matching and S5 homography | V2/V3/V4 benchmarking, Nsight profiling and APOD, slides and presentation material |
| Lê Hà Thanh Chương | 23120195 | S6 warping and blending | V2/V3 verification infrastructure, S6 V4 kernel and tests, final notebook structure and technical analysis |

[Initial stage ownership](docs/report/3.1-team-plan.md) · [Phase assignments and handoffs](docs/task-board.md)

### Runtime and source snapshot

The setup uses a local checkout when opened from the repository and otherwise fetches the pinned public source snapshot. Live timing requires Tesla T4, the CUDA toolkit and the repository build script. The historical evidence cells also run on a CPU-only Python kernel. The snapshot below includes the S6 V4 kernel and the verified V0–V3 sources. Pin the integrated submission SHA before the final Colab run.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nh-atuan/panorama-stitching.git"
SOURCE_REF = "9a887e2777e7b63b9ff76ccf5ed08bbccfb398c1"
WORK_DIR = Path.cwd()
REPO_ROOT = WORK_DIR if (WORK_DIR / "scripts/build.py").is_file() else WORK_DIR / "panorama-stitching"

# Fetch the report sources only when a checkout is unavailable.
if not (REPO_ROOT / "scripts/build.py").is_file():
    if REPO_ROOT.exists():
        raise RuntimeError(f"Expected a repository checkout or a new directory at {REPO_ROOT}.")
    subprocess.run(["git", "init", str(REPO_ROOT)], check=True)
    subprocess.run(["git", "fetch", "--depth", "1", REPO_URL, SOURCE_REF], cwd=REPO_ROOT, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_ROOT, check=True)

DATASET_DIR = Path(os.environ.get("GROUP16_DATASET_DIR", str(REPO_ROOT / "dataset"))).resolve()
BENCHMARK_DIR = Path(
    os.environ.get("GROUP16_BENCHMARK_DIR", str(REPO_ROOT / "results" / "final"))
).resolve()
BASELINE_VERSIONS = (0, 1, 2, 3)
TIMED_RUNS = 5

source = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True, check=False
)
print(f"Source checkout: {source.stdout.strip() or 'local source package (no Git metadata)'}")
print(f"Timed runs per configuration: {TIMED_RUNS}")

for tool, args in (
    ("nvidia-smi", ["--query-gpu=name,driver_version", "--format=csv,noheader"]),
    ("nvcc", ["--version"]),
):
    executable = shutil.which(tool)
    if executable is None:
        print(f"{tool} unavailable in this runtime.")
        continue
    result = subprocess.run([executable, *args], capture_output=True, text=True, check=False)
    print(result.stdout.strip() if result.returncode == 0 else f"{tool} unavailable: {result.stderr.strip()}")

### Dữ liệu và lượt benchmark chung

Ô dưới tải bộ dataset cuối kỳ từ [thư mục Drive](https://drive.google.com/drive/folders/1_J_q5q-K1g7ZNnczTWvXBzwpOsof9ghL) (xem [hướng dẫn dataset](docs/dataset.md)), kiểm tra SHA-256 của `manifest.csv` và của từng ảnh, rồi build `stitch` bằng `scripts/build.py`. Tiếp theo, ô này hỏi chính file `stitch` vừa build xem nó nhận những phiên bản nào và chạy `scripts/run_benchmark.py` một lần cho tất cả các phiên bản đó, có cả cặp challenge. Kết quả được ghi vào `BENCHMARK_DIR`.

Các bảng và biểu đồ ở mục 4 đến mục 8 đều đọc lại từ thư mục này. Nhờ vậy mọi phiên bản được đo trên cùng một GPU, trong cùng một phiên Colab. Code của các bước nằm trong [`scripts/final_notebook.py`](scripts/final_notebook.py).

In [ ]:
import sys

sys.path.insert(0, str(REPO_ROOT / "scripts"))
import final_notebook as fn
from IPython.display import Image, Markdown, display

DATASET_DIR, verified_images = fn.fetch_dataset(DATASET_DIR, WORK_DIR / "dataset_drive")
print(f"Dataset: {DATASET_DIR} ({len(verified_images)} images match manifest.csv)")
STITCH = fn.build_stitch(REPO_ROOT)
run_versions = fn.supported_versions(STITCH)
missing_versions = sorted(set(BASELINE_VERSIONS) - set(run_versions))
if missing_versions:
    raise RuntimeError(f"This build does not accept versions {missing_versions}")
print("Versions in this build:", ", ".join(f"V{version}" for version in run_versions))

if (BENCHMARK_DIR / "environment.json").is_file():
    print(f"Reusing the benchmark session in {BENCHMARK_DIR}")
else:
    fn.run_benchmark_session(REPO_ROOT, STITCH, DATASET_DIR, BENCHMARK_DIR, run_versions, TIMED_RUNS)

benchmark = fn.load_benchmark(BENCHMARK_DIR, run_versions)
benchmark_summary = benchmark.summary
benchmark_speedup = benchmark.speedup
benchmark_failures = benchmark.failures
benchmark_environment = benchmark.environment
CHART_DIR = BENCHMARK_DIR.with_name(BENCHMARK_DIR.name + "_charts")
print(f"GPU: {benchmark_environment['gpu']} | measured at {benchmark_environment['time_utc']} | "
      f"timed runs per configuration: {benchmark_environment['runs']}")

<a id="introduction"></a>

## 2. Introduction

We stitch two overlapping RGB photographs into a panorama. Image A defines the reference frame. The homography $H$ maps points from image B into A, and the output canvas encloses A and the transformed extent of B. A single homography is most suitable for approximately planar scenes or camera rotation with limited translation. Depth changes and moving objects can leave residual misalignment.

```text
Image A → preprocess → detect → describe ↘
                                           match → estimate H → warp & blend → panorama
Image B → preprocess → detect → describe ↗
```

The two feature branches are independent until matching. The estimated homography and the original RGB images feed the renderer. Geometry, panorama fidelity and latency are evaluated separately because matching output pixels across versions establishes implementation parity for a test case, while alignment quality still depends on the estimated geometry and scene.

<a id="chosen-methods"></a>

## 3. Chosen Methods

| Stage | Method | Reason and limitations |
| --- | --- | --- |
| S1 · Preprocessing | Convert interleaved RGB to normalized grayscale using BT.601 luma coefficients, then apply separable Gaussian smoothing with clamped borders. | Smoothing reduces noise before detection. Horizontal and vertical passes use independent pixel stencils. [S1 notes](docs/report/3.3-s1-preprocessing.md) |
| S2 · Detection | FAST segment test, non-maximum suppression and deterministic top-K selection. | Corner tests are local and suit per-pixel execution. NMS limits redundant corners and the cap bounds downstream matching cost. [Rosten and Drummond](https://www.edwardrosten.com/work/rosten_2006_machine.pdf) · [S2 notes](docs/report/3.3-s2-detection.md) |
| S3 · Description | Axis-aligned BRIEF on a separately smoothed grayscale image, with a fixed team-generated pattern and a support-window check. | Binary intensity tests are compact and inexpensive to match. The fixed orientation limits robustness to large in-plane rotation. [Calonder et al.](https://www.cs.ubc.ca/~lowe/525/papers/calonder_eccv10.pdf) · [S3 notes](docs/report/3.3-s3-description.md) |
| S4 · Matching | Exhaustive Hamming search from A to B. Keep the two best distances and apply the ratio test. Cross-check is disabled in the production configuration. | The ratio rejects ambiguous near ties. Each A descriptor can be processed independently, with work proportional to the product of descriptor counts. [Lowe](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf) · [S4 notes](docs/report/3.3-s4-matching.md) |
| S5 · Homography | Seeded RANSAC with minimal normalized DLT, MSAC scoring under forward B→A reprojection error, deterministic stopping and inlier refitting. | Robust estimation rejects many incorrect correspondences. The inlier threshold scales with the diagonal of A. Repeated structures and degenerate samples can still produce an unusable model. [Brown and Lowe](https://www.cs.ubc.ca/~lowe/papers/07brown.pdf) · [S5 notes](docs/report/3.3-s5-homography.md) |
| S6 · Warp and blend | Fit a bounded canvas, map each canvas pixel back into B with $H^{-1}$, sample B with manual bilinear interpolation and average RGB equally where both images cover the pixel. | Inverse mapping avoids forward splatting holes. Coverage is geometric, including valid black pixels. Pixels outside both images are black. [Szeliski](https://szeliski.org/Book/) · [S6 notes](docs/report/3.3-s6-warp-blend.md) |

For inhomogeneous projection $\pi(x,y,w)=(x/w,y/w)$, the forward mapping is $p_A=\pi(H\tilde p_B)$. In S6 the canvas offset is removed before applying the inverse. Sampling checks coordinates against source bounds and clamps bilinear neighbors at the final row and column. Output RGB is quantized after interpolation and blending.

In [ ]:
def show_source(relative, identifiers):
    """Print the selected production statements and identify their source."""
    lines = (REPO_ROOT / relative).read_text(encoding="utf-8").splitlines()
    print(f"\n{relative}")
    for item in identifiers:
        matches = [(number, line) for number, line in enumerate(lines, 1) if item in line]
        if not matches:
            raise ValueError(f"{relative}: missing source statement {item!r}")
        number, line = matches[0]
        print(f"{number:>4}  {line.strip()}")

show_source("src/common/v0_config.h", (
    "kV0GaussianSigma =", "kV0GaussianRadius =", "kV0FastThreshold =", "kV0MaxKeypoints =",
))
show_source("src/common/fast_corner.h", ("kFastArcLength =",))
show_source("src/common/brief_pattern.h", (
    "kBriefTests =", "kBriefPatchRadius =", "kBriefSmoothSigma =", "kBriefSmoothRadius =",
))
show_source("src/common/feature_support.h", (
    "kRatio =", "kMaxIterations =", "kConfidence =", "kSeed =", "kReferenceThreshold =",
))
show_source("src/common/feature_support.cpp", ("candidate.match_cross_check =", "std::hypot(1920.0"))
show_source("src/common/s6_cuda_setup.cuh", (
    "kCoordinateEpsilon =", "kMaxCanvasDimension =", "kMaxCanvasPixels =",
))

<a id="v0-sequential"></a>

## 4. V0 Sequential

V0 executes the six stages in a single C++ thread. The two images are processed in sequence within the feature stages. Descriptors and matches are stored on the host, and S6 traverses the canvas in row-major order. The CLI constructs the shared configuration from image A, decodes both inputs before timing, and encodes the panorama after timing. V0 uses a fixed sampling seed, stable ordering for feature and model selection, and `std::chrono::high_resolution_clock` for timing.

V0 is the reference for output geometry and RGB parity. The midterm run found S4 dominant at small and medium sizes. At large size, the dominant stage shifted to S2 or S6, depending on the site. Matching workload depends on the number of descriptors, while canvas rendering scales with the canvas area and with the number of valid samples from B. [V0 implementation](src/v0/v0_pipeline.cpp) · [Historical midterm timing](docs/report/3.5-timing.md)

In [ ]:
# Historical tables are read by exact header to keep incompatible sessions separate.
from math import exp, fsum, log

def report_table(relative, first_header):
    """Read one repository Markdown table selected by its leading headers."""
    def split(line):
        return tuple(part.strip() for part in line.strip().strip("|").split("|"))

    lines = (REPO_ROOT / relative).read_text(encoding="utf-8").splitlines()
    for index, line in enumerate(lines):
        header = split(line)
        if not line.startswith("|") or header[:len(first_header)] != first_header:
            continue
        rows = []
        for record in lines[index + 2:]:
            if not record.startswith("|"):
                break
            values = split(record)
            rows.append(dict(zip(header, values, strict=True)))
        if not rows:
            raise ValueError(f"No records in {relative}: {first_header}")
        return rows
    raise ValueError(f"Table header not found in {relative}: {first_header}")

V0_REPORT = "docs/report/3.5-timing.md"
v0_timing = report_table(V0_REPORT, ("Site", "Kích thước", "S1", "S2"))
stage_columns = tuple(f"S{stage}" for stage in range(1, 7))
print(f"Historical V0 timing, ms · {V0_REPORT}")
print("Case                S1       S2       S3       S4       S5       S6    Total   Hot")
for row in v0_timing:
    case = f"{row['Site']}/{row['Kích thước']}"
    hot = max(stage_columns, key=lambda stage: float(row[stage]))
    times = " ".join(f"{float(row[col]):8.3f}" for col in (*stage_columns, "Total"))
    print(f"{case:15s} {times}   {hot}")

show_source("src/v0/v0_pipeline.cpp", ("v0_preprocess(", "v0_detect(", "v0_describe(",
                                      "v0_match(", "v0_homography(", "v0_warp_blend("))

### Thời gian V0 trong lượt benchmark chung

Bảng thứ nhất là thời gian trung bình của từng stage, tính bằng ms, cho mọi cấu hình của lượt benchmark ở mục 1. Cột cuối ghi stage chiếm nhiều thời gian nhất và tỉ lệ của nó trong `Total_ms`. Bảng thứ hai lấy trung bình tỉ lệ của mỗi stage trên các site chuẩn, tách theo độ phân giải. Ảnh cuối là panorama V0 của site1 ở độ phân giải small, sinh ra trong cùng lượt chạy.

In [ ]:
display(Markdown(fn.stage_table(benchmark, "V0")))
display(Markdown(fn.dominant_stage_table(benchmark, "V0")))
display(Image(data=fn.panorama_preview(benchmark, "V0", "site1", "small")))

<a id="v1-basic-cuda"></a>

## 5. V1 Basic CUDA

V1 keeps images, feature buffers, matches and the homography in the GPU workspace across stages. Input RGB is transferred to the device, intermediate image and feature buffers are shared between stage calls, and the completed canvas is transferred to the host. Control metadata, including the S6 canvas dimensions required for allocation, is read back separately. Synchronization for metadata, launch overhead and serial control steps can reduce the benefit of parallel kernels.

| Stage | V1 mapping |
| --- | --- |
| S1 | Two-dimensional pixel grid for grayscale and for each separable blur pass. |
| S2 | Pixel threads score FAST corners and suppress local non-maxima. Candidate compaction and deterministic sorting produce the top-K keypoints. |
| S3 | A linear grid processes candidate keypoints. A scan assigns compacted output positions, and keypoint threads construct BRIEF descriptors using the fixed pattern in constant memory. |
| S4 | A block searches B for each A descriptor using Hamming popcount. Reduction keeps the two closest candidates, then prefix and scatter steps pack the accepted matches. |
| S5 | Blocks compute and score batches of hypotheses. A serial commit preserves the deterministic prefix and stopping rule, and the baseline refit uses a single thread. |
| S6 | Setup determines canvas geometry. A pixel grid copies A and a canvas grid gathers B and blends overlap after clearing the canvas. |

GPU stage and transfer timing use CUDA events. `Total_ms` is the synchronized end-to-end pipeline interval, covering H2D, stages, D2H and host-side orchestration within that interval. `Total_Kernel_ms` is the launch-instrumented aggregate of kernel intervals. Stage intervals can include setup work or synchronization, so their sum and the sum of kernel and explicit transfer fields answer different questions. Disk I/O is outside the measurement. Each configuration is warmed up before the timed runs. [V1 pipeline](src/v1/v1_pipeline.cu) · [Timing interface](docs/interface.md) · [CUDA timing guidance](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-c-best-practices-guide/index.html#timing)

In [ ]:
P8_BENCHMARK = "docs/report/p8-t03-v3-apod.md"
P8_VERIFICATION = "docs/report/p8-v3-verification.md"
p8_totals = report_table(P8_BENCHMARK, ("Site", "Size", "V0 ms", "V1 ms"))
p8_real = report_table(P8_VERIFICATION, ("Cấu hình", "V0/V1/V2 tại P7"))

print("Historical P8 comparison, V0 and V1 measured in the same Colab Tesla T4 session")
print("Case                V0 ms     V1 ms   V0/V1")
for row in p8_totals:
    case = f"{row['Site']}/{row['Size']}"
    v0, v1 = float(row["V0 ms"]), float(row["V1 ms"])
    print(f"{case:15s} {v0:9.3f} {v1:9.3f} {v0 / v1:7.2f}x")

print(f"Real-image status parity across V0–V3 reported for {len(p8_real)} configurations.")
show_source("src/v1/v1_pipeline.cu", ("ErrorCode run_gpu_stages(",
                                      "v1_preprocess, v1_detect", "trace.stage_start[i].record"))

### Thời gian V1 trong lượt benchmark chung

Bảng đầu tách thời gian của V1 theo từng stage, kèm H2D, D2H và tổng thời gian kernel. Bảng sau so `Total_ms` và tổng kernel của V1 với V0 trên cùng cấu hình. Ô speedup để trống khi phiên bản dùng làm mốc không chạy được cấu hình đó. Kết quả đối chiếu V1 với V0 (homography, canvas và sai lệch RGB) được ghi trong [tài liệu tích hợp V1](docs/interface.md#118-pipeline-v1-tích-hợp-p6-t01).

In [ ]:
display(Markdown(fn.stage_table(benchmark, "V1")))
display(Markdown(fn.speedup_table(benchmark, "V1")))

<a id="optimized-versions"></a>

## 6. Optimized Versions

The following results come from phase-specific reports with explicit comparison baselines. Section 7 is reserved for the integrated submission run. For a sampled overlap point $q$ in B, the synthetic error is $e(q)=\|\pi(Hq)-\pi(H_{gt}q)\|_2$. The suite checks the root mean square and maximum of these errors. Cross-version drift instead compares the projections of two estimated homographies. The reported maxima describe the sampled points. Acceptance checks status first, then homography, exact canvas dimensions and offsets, and RGB in successful cases. [Verification thresholds and test support](tests/gpu/pipeline_test_support.cuh)

### V2 · Parallel RANSAC scoring and refit

#### Bottleneck

Production full-pipeline timing put S5 ahead of the other stages in V1. Hypothesis construction, scoring and inlier refitting limited progress even while upstream image and descriptor kernels ran on the GPU. The [P7 APOD report](docs/report/p7-t03-v2-apod.md) explains why isolated profiler measurements alone did not establish this ranking.

#### Idea

Distribute hypothesis evaluation and all-inlier refitting across threads. Preserve the ordered decision about which hypotheses are committed so speculative parallel computation cannot change the RANSAC prefix.

#### Implementation

V2 replaces the S5 call and reuses V1 for S1–S4 and S6. Threads cooperate on Jacobi SVD operations and MSAC scoring. Refit kernels process inlier contributions in parallel and use shared working data. The seed, normalized DLT, reprojection threshold and inlier-selection policy remain the reference configuration. [V2 S5 source](src/v2/s5/v2_homography.cu) · [V2 pipeline](src/v2/v2_pipeline.cu)

#### Result

P7 compared V2 with V1 in the same T4 session. Its V2/V0 ratio used the earlier CPU baseline. The P8 experiment later reran both V0 and V2 together, so an end-to-end V2/V0 comparison from P8 is shown separately below. The acceptance run verified the synthetic thresholds, real-image status parity and exact RGB agreement on the successful panoramas. [P7 acceptance](docs/report/p7-v2-verification.md)

#### Analysis

The largest pipeline gains occurred where S5 previously dominated. S5 remained the largest stage across the standard cases in P7, but S6 was already close to S5 at site2/large. The V2 Nsight trace pointed to the hypothesis kernel as a candidate for V3. Register pressure and low achieved occupancy motivated changes to how work was scheduled and reduced.

In [ ]:
def geomean(values):
    values = list(values)
    return exp(fsum(log(value) for value in values) / len(values))

P7_BENCHMARK = "docs/report/p7-t03-v2-apod.md"
P7_VERIFICATION = "docs/report/p7-v2-verification.md"
p7_totals = report_table(P7_BENCHMARK, ("Site", "Size", "V1 P6 total", "V1 P6 S5"))
p7_synthetic = report_table(P7_VERIFICATION, ("Fixture", "RMSE V2 (px)", "Max V2 (px)"))
p7_real = report_table(P7_VERIFICATION, ("Cấu hình", "Trạng thái chung", "Canvas"))
v2_over_v1 = [float(r["V1 P7 total"]) / float(r["V2 total"]) for r in p7_totals]
v2_over_v0_p8 = [float(r["V0 ms"]) / float(r["V2 ms"]) for r in p8_totals]
p7_success = [r for r in p7_real if r["Trạng thái chung"] == "Success"]
print("V2/V1 end-to-end speedup, same-session P7 Tesla T4:"
      f" {min(v2_over_v1):.2f}–{max(v2_over_v1):.2f}x, geometric mean {geomean(v2_over_v1):.2f}x")
print("V2/V0 end-to-end speedup, independent same-session P8 repeat:"
      f" {min(v2_over_v0_p8):.2f}–{max(v2_over_v0_p8):.2f}x")
print(f"P7 verification: {len(p7_synthetic)} synthetic cases and {len(p7_real)} real status comparisons.")
print(f"Max RGB difference across {len(p7_success)} successful panoramas:"
      f" {max(int(r['Max RGB']) for r in p7_success)} levels/channel.")

### V3 · RANSAC kernel scheduling and reductions

#### Bottleneck

P7 profiling identified `compute_hypotheses` as the dominant V2 kernel in the site1 traces. Register pressure limited occupancy in the profiled launches, and recurring batch launches added scheduling overhead.

#### Idea

Use warp shuffle for the MSAC reduction, share DLT and SVD scratch storage across the block, and process larger batches. The optimization targets more active work on each multiprocessor and fewer launches while preserving the ordered hypothesis commit.

#### Implementation

V3 updates S5 in the shared GPU pipeline. The hypothesis kernel uses shared arrays for the design matrix and right singular vectors, warp-level reduction for score accumulation, and explicit launch bounds. Batch-size changes reduce the number of hypothesis launches. S1–S4 and S6 reuse V1. [V3 S5 source](src/v3/s5/v3_homography.cu) · [V3 pipeline](src/v3/v3_pipeline.cu) · [CUDA Best Practices Guide](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-c-best-practices-guide/index.html)

#### Result

In the same-session P8 Tesla T4 benchmark, V3 improved end-to-end time in **8 of 9** standard configurations relative to V2, with a **1.85×** geometric-mean speedup. The denominator is V3's average `Total_ms` in each configuration. The corresponding V0-relative ratio uses the V0 timing from the same P8 run. The S5-only ratio uses the per-stage CUDA-event interval. [P8 benchmark and profiling](docs/report/p8-t03-v3-apod.md#4-result)

P8 acceptance separately confirmed the synthetic thresholds and real-image parity. It ran Compute Sanitizer on the V3 suite and a CLI case. The test and sanitizer timings are excluded from benchmark comparisons. [P8 acceptance](docs/report/p8-v3-verification.md)

#### Analysis

V3 was slightly slower at site2/medium, where S5 timing was nearly unchanged. At site2/large, S6 reached **28.468 ms**, **45.7%** of V3 `Total_ms`. That shift motivated V4. Nsight reported fewer hypothesis launches at site1 and higher achieved occupancy in the sampled site1/medium launch. Barrier-stall samples remained substantial. The experiment evaluated the optimization as a package, so it does not isolate the effect of each change.

In [ ]:
p8_stage = report_table(P8_BENCHMARK, ("Site", "Size", "S5 V2 ms", "S5 V3 ms"))
p8_synthetic = report_table(P8_VERIFICATION, ("Ca", "Điểm Q theo P7"))
p8_memory = report_table(P8_VERIFICATION, ("Target", "Exit", "Errors", "Leaked bytes"))
show_source("tests/gpu/pipeline_test_support.cuh", ("kGrid =", "kTruthRmsePx =",
              "kTruthMaxPx =", "kDriftPx =", "kRgbTolerance ="))

ratios_v2 = [float(r["V2 ms"]) / float(r["V3 ms"]) for r in p8_totals]
ratios_v0 = [float(r["V0 ms"]) / float(r["V3 ms"]) for r in p8_totals]
ratios_s5 = [float(r["S5 V2 ms"]) / float(r["S5 V3 ms"]) for r in p8_stage]
v3_wins = sum(ratio > 1 for ratio in ratios_v2)
print("Historical P8 benchmark · Colab Tesla T4")
print(f"V3/V2 end-to-end: {v3_wins}/{len(ratios_v2)} faster,"
      f" geometric mean {geomean(ratios_v2):.2f}x.")
print(f"V3/V0 end-to-end: geometric mean {geomean(ratios_v0):.2f}x.")
print(f"V3/V2 S5 stage: geometric mean {geomean(ratios_s5):.2f}x.")

target = ("site2", "large")
large_total = next(r for r in p8_totals if (r["Site"], r["Size"]) == target)
large_stage = next(r for r in p8_stage if (r["Site"], r["Size"]) == target)
s6_ms = float(large_stage["S6 V3 ms"])
print(f"site2/large: S6 {s6_ms:.3f} ms of {float(large_total['V3 ms']):.3f} ms total"
      f" ({s6_ms / float(large_total['V3 ms']):.1%}).")

synthetic_errors = [
    tuple(float(value.removesuffix(" px")) for value in
          row["V3 RMSE / Max / drift / canvas / RGB"].split(" / ")[:2])
    for row in p8_synthetic
]
print(f"P8 acceptance: {len(synthetic_errors)} synthetic cases,"
      f" worst RMSE {max(rmse for rmse, _ in synthetic_errors):.3g} px,"
      f" worst point error {max(error for _, error in synthetic_errors):.3g} px.")
p8_real_status = "Trạng thái V0–V3, drift, canvas và RGB của lượt P8"
real_success = [row[p8_real_status] for row in p8_real if row[p8_real_status].startswith("Success")]
max_rgb = max(int(status.rsplit(" ", 1)[1]) for status in real_success)
print(f"Status parity reported for {len(p8_real)} real configurations,"
      f" Max RGB {max_rgb} across {len(real_success)} successful panoramas.")
for row in p8_memory:
    print(f"Memcheck {row['Target']}: exit {row['Exit']}, errors {row['Errors']},"
          f" leaked bytes {row['Leaked bytes']}.")

### V4 · Single-pass Canvas Fusion

#### Bottleneck

At site2/large, P8 timing ranked S6 ahead of S5. The V1 renderer used by V3 clears the canvas, copies A and then traverses the canvas again to sample B and blend. The overlap path reads back the canvas value written by the copy step. [P8 analysis](docs/report/p8-t03-v3-apod.md#5-analysis)

#### Idea

Use a single rendering pass. Assign one thread to each canvas pixel, determine source coverage directly and write the final RGB once. Fusing canvas clear, copy and gather reduces launches and intermediate canvas traffic. Setup and output transfer remain separate.

#### Implementation

V4 uses the shared CUDA setup to check homography validity and derive canvas geometry. The fused kernel uses a two-dimensional grid, reads A directly where covered, maps the output coordinate into B and computes manual bilinear sampling. It preserves the V1 sampling order and performs the same equal-weight average before quantization. Out-of-coverage pixels are written as black. The device workspace retains ownership of the rendered canvas until the pipeline retrieves it. [V4 S6 source](src/v4/s6/v4_warp_blend.cu) · [V4 experiment plan](docs/report/v4-v5-optimization-plan.md)

#### Result

**Pending V4 benchmark evidence · Đế (P9-T05).** Insert the verified T4 stage and end-to-end timings with speedup against both V0 and V3. Add the integrated parity and memory-safety outcome from Tuấn (P9-T02). The merged S6 kernel has separate same-homography component tests. The integrated timing and verification belong to the production V4 pipeline. [S6 tests and results](docs/stages/s6-warp-blend.md) · [S6 merge](https://github.com/nh-atuan/panorama-stitching/pull/41)

#### Analysis

Compare S6 time alongside total pipeline time, first at site2/large and then across the full standard matrix. Inspect kernel launch counts, memory transactions or bytes, and the effect of canvas size on the timing. Report any regressions and relate the result to canvas coverage, sampling cost, and the remaining S5 and transfer time. Verify status and geometry before checking pixel parity.

In [ ]:
v4_summary = BENCHMARK_DIR / "v4" / "summary.csv"
if v4_summary.is_file():
    print(f"V4 benchmark candidate found: {v4_summary.relative_to(BENCHMARK_DIR)}")
    print("Check the session, production checkout and acceptance evidence before reporting speedup.")
else:
    print("V4 benchmark pending. The historical analysis above remains available.")
show_source("src/v4/s6/v4_warp_blend.cu", ("const bool has_a =", "const double blended =",
                                          "const dim3 block(", "render_canvas_kernel<<<"))

#### Kết quả V4 trong lượt benchmark chung

Khi bản build nhận `--version 4`, V4 được đo cùng lượt với các phiên bản khác. Ô dưới in thời gian từng stage của V4 và speedup của nó so với V0 và V3.

In [ ]:
if "V4" in benchmark.versions:
    display(Markdown(fn.stage_table(benchmark, "V4")))
    display(Markdown(fn.speedup_table(benchmark, "V4")))
else:
    print("V4 is not part of this benchmark session.")

<a id="final-comparison"></a>

## 7. Final Comparison

For version $v$, compute end-to-end speedup against V0 as $T_0/T_v$ and incremental speedup as $T_{v-1}/T_v$, using the mean times for the same pair and resolution in one Colab session. Report transfer and aggregate kernel intervals separately. Show the six stage intervals for the best-performing version by configuration or under an explicitly stated aggregation rule. Failed challenge cases have a status and log, with no speedup assigned.

In [ ]:
env = benchmark.environment
print(f"GPU: {env['gpu']} | CPU: {env['cpu']} | measured at {env['time_utc']} | timed runs: {env['runs']}")
display(Markdown(fn.speedup_table(benchmark)))
display(Markdown(fn.geomean_table(benchmark)))
display(Markdown(fn.failure_table(benchmark)))

Biểu đồ đầu tiên là speedup end-to-end so với V0, nhóm theo site và độ phân giải. Mỗi biểu đồ sau đó tách speedup theo stage cho một phiên bản GPU. Phiên bản tốt nhất là phiên bản có trung bình nhân speedup so với V0 lớn nhất trên các cấu hình chuẩn. Bảng và biểu đồ cột chồng cuối mục cho thấy thời gian từng stage của phiên bản đó.

In [ ]:
for chart in fn.plot_speedup_charts(benchmark, CHART_DIR):
    display(Image(filename=str(chart)))
best_version = fn.best_version(benchmark)
print(f"Best version (geometric-mean speedup vs V0 over the standard configurations): {best_version}")
display(Markdown(fn.stage_table(benchmark, best_version)))
display(Image(filename=str(fn.plot_stage_breakdown(benchmark, best_version, CHART_DIR))))

<a id="results-gallery"></a>

## 8. Results Gallery

Show the input pair beside its panorama for the provided sample and the team's own standard sites. Include the challenge case with the two source views and either its distorted panorama or its recorded failure state.

### Challenge failure analysis

P8 recorded a panorama at small resolution, with the railing doubled in the overlap and a strong projective distortion in image B. Medium and large returned `NumericalFailure` from S6. The logs identify the reporting stage. A specific geometry check inside S6 would require additional diagnostics. Equality across versions leaves this visible mismatch intact. A single projective transform and equal-weight averaging can leave residual errors in repetitive, low-light scenes. [P8 challenge observations](docs/report/p8-v3-verification.md#7-challenge-observations)

#### Ảnh đầu vào và panorama

Mỗi hàng gồm ảnh A, ảnh B và panorama do phiên bản tốt nhất ở mục 7 tạo ra trong lượt benchmark. Với cấu hình bị lỗi, ô panorama hiện mã lỗi và vài dòng cuối của log. Hàng cuối là cặp challenge ở độ phân giải small, nơi pipeline vẫn ra ảnh nhưng ảnh bị méo. Ảnh mẫu do môn học cung cấp chưa có trong repo nên chưa có mặt trong gallery.

In [ ]:
gallery = fn.gallery_entries(benchmark, DATASET_DIR, best_version, "medium")
gallery += fn.gallery_entries(benchmark, DATASET_DIR, best_version, "small", sites=("challenge",))
display(Image(filename=str(fn.plot_gallery(gallery, CHART_DIR / f"gallery_{best_version.lower()}.jpg"))))
for entry in gallery:
    print(f"{entry['site']}/{entry['size']} {entry['version']}: {entry['status']}")

In [ ]:
print("Historical P8 challenge outcomes")
for row in p8_real:
    if row["Cấu hình"].startswith("challenge/"):
        print(f"{row['Cấu hình']}: {row[p8_real_status]}")

<a id="conclusion"></a>

## 9. Conclusion

Moving the pipeline to CUDA changed its dominant costs. V1 made the S5 bottleneck visible, and V2 and V3 improved it by reorganizing hypothesis work while preserving the verified output. The V3 measurements exposed a different bottleneck at site2/large, where the canvas renderer becomes the next optimization target.

The verified regressions establish parity for these fixtures and datasets. Camera translation, limited overlap, repeated structure and radiometric differences still affect the quality of the underlying algorithm. A next investigation could compare observed matches, geometric residuals and seam quality for difficult inputs. Keep such algorithm changes in a separate experiment with their own accuracy criteria.

The submission run will close the V4 analysis using same-session timing, profiling and acceptance evidence, and will link the presentation video here once published.

<a id="references"></a>

## 10. References

### Methods and implementation

- ITU-R. *Recommendation BT.601*, luma coefficients used in S1. [Recommendation](https://www.itu.int/rec/R-REC-BT.601)
- Rosten, E., and Drummond, T. (2006). *Machine Learning for High-Speed Corner Detection*. ECCV. [Author PDF](https://www.edwardrosten.com/work/rosten_2006_machine.pdf)
- Calonder, M., Lepetit, V., Strecha, C., and Fua, P. (2010). *BRIEF: Binary Robust Independent Elementary Features*. ECCV. [Paper](https://www.cs.ubc.ca/~lowe/525/papers/calonder_eccv10.pdf)
- Lowe, D. G. (2004). *Distinctive Image Features from Scale-Invariant Keypoints*. IJCV. The ratio test informs S4. [Author PDF](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf)
- Brown, M., and Lowe, D. G. (2007). *Automatic Panoramic Image Stitching using Invariant Features*. IJCV. [Author PDF](https://www.cs.ubc.ca/~lowe/papers/07brown.pdf)
- Szeliski, R. (2022). *Computer Vision: Algorithms and Applications*, second edition. Image alignment, resampling and compositing. [Author site](https://szeliski.org/Book/)
- Vigna, S. *SplitMix64 reference implementation*, used as the S5 sampler's mixer reference. [Reference code](https://prng.di.unimi.it/splitmix64.c)
- NVIDIA. *CUDA C++ Best Practices Guide* and *Nsight Systems User Guide*. [CUDA guidance](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-c-best-practices-guide/index.html) · [Nsight Systems](https://docs.nvidia.com/nsight-systems/UserGuide/index.html)
- Barrett, S., and contributors. `stb_image` and `stb_image_write`, the vendored image I/O headers. [stb repository](https://github.com/nothings/stb) · [Stage source attributions](docs/report/references.md)

### Evidence and reproducibility

- [Assignment](docs/specs/CSC14120_2026_Final%20Project.md) · [Build and CLI](README.md) · [Official dataset](docs/dataset.md) · [Results layout](docs/results-guide.md)
- [P7 V2 benchmark](docs/report/p7-t03-v2-apod.md) · [P7 V2 verification](docs/report/p7-v2-verification.md) · [P8 V3 benchmark](docs/report/p8-t03-v3-apod.md) · [P8 V3 verification](docs/report/p8-v3-verification.md)
- [Submission run and pending integrations](docs/report/p9-final-notebook-handoff.md)

Run the notebook in order from the source and runtime setup through the final comparison and gallery. The final Colab run will retain the output cells, source SHA, dataset manifest, GPU and toolkit identity, and benchmark provenance. P7 and P8 evidence in the current structure remains associated with its original measurement sessions.

### AI tool usage

OpenAI Codex assisted with drafting and organizing this notebook, checking technical descriptions against repository code and reports, and validating the notebook structure and runnable cells. The cited experiment reports supply the measurements. The team will consolidate any additional AI usage from other contributors in the submitted version.